In [1]:
# Step 1: Imports
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

os.environ["TRANSFORMERS_NO_TF"] = "1"  # Prevent TensorFlow backend

# Step 2: Load and preprocess full dataset
csv_path = "C:/Users/DELL/Desktop/Capstone Project/financial_phrasebank_with_emotions.csv"
df = pd.read_csv(csv_path)

# Define label columns
emotion_labels = ['anger', 'anticipation', 'disgust', 'fear', 'joy',
                  'negative', 'positive', 'sadness', 'surprise', 'trust']

# ✅ Fix: Convert label counts to binary (0 or 1)
df[emotion_labels] = (df[emotion_labels] > 0).astype(int)

# Step 3: Train-test split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["sentence"].tolist(), df[emotion_labels].values, test_size=0.2, random_state=42
)

# Step 4: Tokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base", use_fast=True)

# Step 5: Dataset Class
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

train_dataset = EmotionDataset(train_texts, train_labels, tokenizer)
val_dataset = EmotionDataset(val_texts, val_labels, tokenizer)

# Step 6: Load model
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(emotion_labels),
    problem_type="multi_label_classification"
)

# Step 7: Training Arguments
training_args = TrainingArguments(
    output_dir="./roberta_emotion_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",  # 🧠 use macro f1 to optimize rare labels too
    report_to=[]
)

# Step 8: Evaluation Metric Function
def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs > 0.5).astype(int)
    label_ids = np.array(label_ids).astype(int)

    assert preds.shape == label_ids.shape, f"Shape mismatch: {preds.shape} vs {label_ids.shape}"

    report = classification_report(
        label_ids,
        preds,
        target_names=emotion_labels,
        output_dict=True,
        zero_division=0
    )

    return {
        **{f"{label}_f1": report[label]["f1-score"] for label in emotion_labels},
        "micro_f1": report["micro avg"]["f1-score"],
        "macro_f1": report["macro avg"]["f1-score"]
    }

# Step 9: Trainer Setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

# Step 10: Train the model
trainer.train()

# Step 11: Save the best model and tokenizer
trainer.save_model("./roberta_emotion_model")
tokenizer.save_pretrained("./roberta_emotion_model")
print("✅ Model and tokenizer saved to ./roberta_emotion_model")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\DELL\anaconda3\Lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\DELL\AppData\Local\Temp\ipykernel_3724\148003696.py:102: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Anger F1,Anticipation F1,Disgust F1,Fear F1,Joy F1,Negative F1,Positive F1,Sadness F1,Surprise F1,Trust F1,Micro F1,Macro F1
1,No log,0.334610,0.475000,0.433333,0.000000,0.552632,0.486486,0.533333,0.720000,0.652632,0.000000,0.575342,0.577931,0.442876
2,No log,0.286993,0.562500,0.590308,0.000000,0.571429,0.671533,0.630769,0.778243,0.781609,0.000000,0.710227,0.681963,0.529662
3,0.361300,0.269488,0.600000,0.580357,0.000000,0.600000,0.695652,0.602941,0.795556,0.741573,0.320000,0.673469,0.679677,0.560955
4,0.361300,0.266931,0.612903,0.641026,0.000000,0.603175,0.684932,0.662162,0.798186,0.787234,0.384615,0.690554,0.700196,0.586479


✅ Model and tokenizer saved to ./roberta_emotion_model


In [3]:
from transformers import pipeline

classifier = pipeline("text-classification", model="./roberta_emotion_model", tokenizer="./roberta_emotion_model", return_all_scores=True)

result = classifier("The company achieved record profits and received praise from investors.")
for r in result[0]:
    print(f"{r['label']}: {r['score']:.2f}")


Device set to use cpu


LABEL_0: 0.03
LABEL_1: 0.13
LABEL_2: 0.02
LABEL_3: 0.03
LABEL_4: 0.05
LABEL_5: 0.08
LABEL_6: 0.29
LABEL_7: 0.03
LABEL_8: 0.02
LABEL_9: 0.14


C:\Users\DELL\anaconda3\Lib\site-packages\transformers\pipelines\text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [5]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("./roberta_emotion_model")


In [7]:
emotion_labels = ['anger', 'anticipation', 'disgust', 'fear', 'joy',
                  'negative', 'positive', 'sadness', 'surprise', 'trust']

model.config.id2label = {i: label for i, label in enumerate(emotion_labels)}
model.config.label2id = {label: i for i, label in enumerate(emotion_labels)}


In [3]:
from transformers import AutoModelForSequenceClassification

emotion_labels = ['anger', 'anticipation', 'disgust', 'fear', 'joy',
                  'negative', 'positive', 'sadness', 'surprise', 'trust']

model = AutoModelForSequenceClassification.from_pretrained("./roberta_emotion_model")
model.config.id2label = {i: label for i, label in enumerate(emotion_labels)}
model.config.label2id = {label: i for i, label in enumerate(emotion_labels)}

model.save_pretrained("./roberta_emotion_model_labeled")


In [5]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./roberta_emotion_model_labeled",
    tokenizer="./roberta_emotion_model",
    top_k=None
)

text = "Investors responded positively to the record-breaking financial results."
results = classifier(text)

for r in results[0]:
    print(f"{r['label']}: {r['score']:.2f}")


Device set to use cpu


positive: 0.61
trust: 0.33
anticipation: 0.28
joy: 0.09
negative: 0.08
surprise: 0.03
fear: 0.03
anger: 0.03
sadness: 0.03
disgust: 0.02


In [19]:
import os
import pandas as pd
from transformers import pipeline

# Folder paths
input_folder = "C:/Users/DELL/Desktop/New Data"
output_folder = os.path.join(input_folder, "with_emotions")
os.makedirs(output_folder, exist_ok=True)

# Load model once
classifier = pipeline(
    "text-classification",
    model="./roberta_emotion_model_labeled",
    tokenizer="./roberta_emotion_model",
    top_k=None,
    truncation=True
)

# Loop over all CSV files in folder
for file_name in os.listdir(input_folder):
    if file_name.endswith(".csv"):
        input_path = os.path.join(input_folder, file_name)
        output_path = os.path.join(output_folder, file_name.replace(".csv", "_emotions.csv"))

        print(f"🔄 Processing: {file_name}")
        try:
            df = pd.read_csv(input_path)

            # Validate 'content' column
            if "content" not in df.columns:
                print(f"⚠️ Skipped (no 'content' column): {file_name}")
                continue

            # Clean data
            df = df[df["content"].notna()]
            df = df[df["content"].apply(lambda x: isinstance(x, str))].reset_index(drop=True)

            if df.empty:
                print(f"⚠️ Skipped (empty 'content'): {file_name}")
                continue

            # Run inference
            emotion_results = classifier(df["content"].tolist())

            # Convert predictions to DataFrame
            emotion_scores = pd.DataFrame([
                {res["label"]: res["score"] for res in row}
                for row in emotion_results
            ])

            # Merge and save
            df_final = pd.concat([df, emotion_scores], axis=1)
            df_final.to_csv(output_path, index=False)
            print(f"✅ Saved: {output_path}")

        except Exception as e:
            print(f"❌ Error processing {file_name}: {e}")


Device set to use cpu


🔄 Processing: Adobe.csv
✅ Saved: C:/Users/DELL/Desktop/New Data\with_emotions\Adobe_emotions.csv
🔄 Processing: Amazon.csv
✅ Saved: C:/Users/DELL/Desktop/New Data\with_emotions\Amazon_emotions.csv
🔄 Processing: Apple.csv
✅ Saved: C:/Users/DELL/Desktop/New Data\with_emotions\Apple_emotions.csv
🔄 Processing: Bank_of_America.csv
✅ Saved: C:/Users/DELL/Desktop/New Data\with_emotions\Bank_of_America_emotions.csv
🔄 Processing: Berkshire_Hathaway.csv
✅ Saved: C:/Users/DELL/Desktop/New Data\with_emotions\Berkshire_Hathaway_emotions.csv
🔄 Processing: Chevron.csv
✅ Saved: C:/Users/DELL/Desktop/New Data\with_emotions\Chevron_emotions.csv
🔄 Processing: ExxonMobil.csv
✅ Saved: C:/Users/DELL/Desktop/New Data\with_emotions\ExxonMobil_emotions.csv
🔄 Processing: General_Motors.csv
✅ Saved: C:/Users/DELL/Desktop/New Data\with_emotions\General_Motors_emotions.csv
🔄 Processing: Goldman_Sachs.csv
✅ Saved: C:/Users/DELL/Desktop/New Data\with_emotions\Goldman_Sachs_emotions.csv
🔄 Processing: Intel.csv
✅ Saved

In [21]:
pip install yfinance


     ---------------------------------------- 0.0/949.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.0 kB ? eta -:--:--
     - ------------------------------------- 30.7/949.0 kB 1.3 MB/s eta 0:00:01
     --- ----------------------------------- 92.2/949.0 kB 1.1 MB/s eta 0:00:01
     ---------- --------------------------- 256.0/949.0 kB 1.7 MB/s eta 0:00:01
     --------------------- ---------------- 532.5/949.0 kB 2.8 MB/s eta 0:00:01
     -------------------------------------  942.1/949.0 kB 4.0 MB/s eta 0:00:01
     -------------------------------------- 949.0/949.0 kB 3.8 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 

In [29]:
pip install alpha_vantage


Note: you may need to restart the kernel to use updated packages.


In [31]:
from alpha_vantage.timeseries import TimeSeries
import pandas as pd
import time
import os

# Initialize Alpha Vantage client
api_key = "GH7Q32YUWTUCAO38"  # 🔁 Replace this with your actual API key
ts = TimeSeries(key=api_key, output_format='pandas')

# Input/output folders
input_folder = "C:/Users/DELL/Desktop/New Data/with_emotions"
output_folder = input_folder + "_with_prices"
os.makedirs(output_folder, exist_ok=True)

# Ticker map
company_to_ticker = {
    "Adobe": "ADBE",
    "Amazon": "AMZN",
    "Apple": "AAPL",
    "Bank_of_America": "BAC",
    "Berkshire_Hathaway": "BRK.B",  # AV uses '.' not '-'
    "Chevron": "CVX",
    "ExxonMobil": "XOM",
    "General_Motors": "GM",
    "Goldman_Sachs": "GS",
    "Intel": "INTC",
    "Johnson_&_Johnson": "JNJ",
    "JPMorgan_Chase": "JPM",
    "Meta": "META",
    "Microsoft": "MSFT",
    "Netflix": "NFLX",
    "Nvidia": "NVDA",
    "Pfizer": "PFE",
    "Procter_&_Gamble": "PG",
    "Tesla": "TSLA",
    "Walmart": "WMT"
}

# Loop through all files
for filename in os.listdir(input_folder):
    if filename.endswith("_emotions.csv"):
        company = filename.replace("_emotions.csv", "")
        ticker = company_to_ticker.get(company)
        if not ticker:
            print(f"⚠️ Skipping {company} — no ticker.")
            continue

        try:
            df = pd.read_csv(os.path.join(input_folder, filename))
            if "publishedAt" not in df.columns:
                print(f"⚠️ Skipping {filename} — missing 'publishedAt'.")
                continue

            df["publishedAt"] = pd.to_datetime(df["publishedAt"], errors="coerce").dt.date
            df = df.dropna(subset=["publishedAt"])

            if df.empty:
                print(f"⚠️ Skipping {filename} — no valid dates.")
                continue

            print(f"📈 Fetching prices for {ticker}")
            price_df, _ = ts.get_daily(symbol=ticker, outputsize='full')
            price_df = price_df.reset_index()
            price_df["date"] = price_df["date"].dt.date
            price_df = price_df[["date", "4. close"]].rename(columns={"4. close": "Close"})

            # Merge with emotion data
            merged = df.merge(price_df, left_on="publishedAt", right_on="date", how="left")
            merged.drop(columns=["date"], inplace=True)

            # Save
            output_path = os.path.join(output_folder, filename)
            merged.to_csv(output_path, index=False)
            print(f"✅ Saved: {output_path}")

            # Rate limit: 5 calls/minute on free plan
            time.sleep(12)

        except Exception as e:
            print(f"❌ Error processing {company}: {e}")


📈 Fetching prices for ADBE
✅ Saved: C:/Users/DELL/Desktop/New Data/with_emotions_with_prices\Adobe_emotions.csv
📈 Fetching prices for AMZN
✅ Saved: C:/Users/DELL/Desktop/New Data/with_emotions_with_prices\Amazon_emotions.csv
📈 Fetching prices for AAPL
✅ Saved: C:/Users/DELL/Desktop/New Data/with_emotions_with_prices\Apple_emotions.csv
📈 Fetching prices for BAC
✅ Saved: C:/Users/DELL/Desktop/New Data/with_emotions_with_prices\Bank_of_America_emotions.csv
📈 Fetching prices for BRK.B
✅ Saved: C:/Users/DELL/Desktop/New Data/with_emotions_with_prices\Berkshire_Hathaway_emotions.csv
📈 Fetching prices for CVX
✅ Saved: C:/Users/DELL/Desktop/New Data/with_emotions_with_prices\Chevron_emotions.csv
📈 Fetching prices for XOM
✅ Saved: C:/Users/DELL/Desktop/New Data/with_emotions_with_prices\ExxonMobil_emotions.csv
📈 Fetching prices for GM
✅ Saved: C:/Users/DELL/Desktop/New Data/with_emotions_with_prices\General_Motors_emotions.csv
📈 Fetching prices for GS
✅ Saved: C:/Users/DELL/Desktop/New Data/wit

In [35]:
import pandas as pd
import os

input_folder = "C:/Users/DELL/Desktop/New Data/with_emotions_with_prices"
output_folder = "C:/Users/DELL/Desktop/New Data/with_emotions_filled"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):
    if file.endswith(".csv") or file.endswith(".xlsx"):
        input_path = os.path.join(input_folder, file)
        output_path = os.path.join(output_folder, file.replace(".xlsx", ".csv"))

        # Load
        if file.endswith(".xlsx"):
            df = pd.read_excel(input_path)
        else:
            df = pd.read_csv(input_path)

        # Fill Close
        if "Close" in df.columns:
            df["Close"] = df["Close"].ffill().bfill()

        # Save
        df.to_csv(output_path, index=False)
        print(f"✅ Saved filled version: {output_path}")


✅ Saved filled version: C:/Users/DELL/Desktop/New Data/with_emotions_filled\Adobe_emotions.csv
✅ Saved filled version: C:/Users/DELL/Desktop/New Data/with_emotions_filled\Amazon_emotions.csv
✅ Saved filled version: C:/Users/DELL/Desktop/New Data/with_emotions_filled\Apple_emotions.csv
✅ Saved filled version: C:/Users/DELL/Desktop/New Data/with_emotions_filled\Bank_of_America_emotions.csv
✅ Saved filled version: C:/Users/DELL/Desktop/New Data/with_emotions_filled\Berkshire_Hathaway_emotions.csv
✅ Saved filled version: C:/Users/DELL/Desktop/New Data/with_emotions_filled\Chevron_emotions.csv
✅ Saved filled version: C:/Users/DELL/Desktop/New Data/with_emotions_filled\ExxonMobil_emotions.csv
✅ Saved filled version: C:/Users/DELL/Desktop/New Data/with_emotions_filled\General_Motors_emotions.csv
✅ Saved filled version: C:/Users/DELL/Desktop/New Data/with_emotions_filled\Goldman_Sachs_emotions.csv
✅ Saved filled version: C:/Users/DELL/Desktop/New Data/with_emotions_filled\Intel_emotions.csv
✅ 